In [6]:
import shapely.geometry as geom

usda_vertices = {
    "Sand": [(100,0,0),(85,0,15),(90,10,0),(100,0,0)],
    "Loamy Sand": [(85,0,15),(70,0,30),(75,15,10),(90,10,0),(85,0,15)],
    "Sandy Loam": [(70,0,30),(52,0,48),(52,20,28),(45,28,27),(50,50,0),(75,15,10),(70,0,30)],
    "Loam": [(52,20,28),(28,28,44),(22,50,28),(40,40,20),(43,23,34),(45,28,27),(52,20,28)],
    "Silt Loam": [(20,0,80),(0,0,100),(0,12,88),(8,50,42),(28,50,22),(28,28,44),(52,20,28),(43,23,34),(40,40,20),(22,50,28),(20,0,80)],
    "Silt": [(0,0,100),(0,12,88),(0,0,100)],
    "Sandy Clay Loam": [(45,28,27),(43,23,34),(65,0,35),(52,0,48),(52,20,28),(45,28,27)],
    "Clay Loam": [(28,28,44),(22,50,28),(40,40,20),(47,40,13),(55,20,25),(43,23,34),(28,28,44)],
    "Silty Clay Loam": [(28,50,22),(22,50,28),(0,50,50),(0,40,60),(8,50,42),(28,50,22)],
    "Sandy Clay": [(65,0,35),(55,0,45),(45,20,35),(55,20,25),(65,0,35)],
    "Silty Clay": [(0,40,60),(0,12,88),(8,50,42),(0,40,60)],
    "Clay": [(55,0,45),(45,20,35),(40,40,20),(47,40,13),(55,20,25),(55,0,45)]
}

polygons = {k: geom.Polygon([(v[0], v[2]) for v in pts]) for k, pts in usda_vertices.items()}

centroids = {k: (p.centroid.x, p.centroid.y) for k, p in polygons.items()}


In [ ]:
import pandas as pd

# example: centroids = {"Loam": (40.2, 23.8), "Sandy Loam": (65.1, 12.4), ...}

centroid_df = (
    pd.DataFrame.from_dict(centroids, orient="index", columns=["sand_centroid", "clay_centroid"])
    .assign(silt_centroid=lambda d: 100 - d.sand_centroid - d.clay_centroid)
)

# soil_texture column must match centroid keys exactly; normalize if needed
df["soil_texture_norm"] = df["soil_texture"].str.strip().str.title()

df = df.merge(
    centroid_df,
    left_on="soil_texture_norm",
    right_index=True,
    how="left"
)

df = df.drop(columns=["soil_texture_norm"])
